# Lab 1 — Data Preprocessing & Split

This notebook is the **single source of truth** for data preparation. It:

1. Loads and cleans both the **1 K** and **25 K** Amazon review datasets using `preprocess_pandas`.
2. Applies a reproducible **stratified train / validation / test split** (81 % / 9 % / 10 %).
3. Saves each split to CSV so that every downstream experiment notebook (ANN, Transformer, etc.) can load identical data without repeating any of this logic.

**Output files written to `../data/splits/`**

```
splits/
├── 1k_train.csv
├── 1k_val.csv
├── 1k_test.csv
├── 25k_train.csv
├── 25k_val.csv
└── 25k_test.csv
```

Each CSV has two columns: `Sentence` (preprocessed text) and `Class` (0 = negative, 1 = positive).

> **Run this notebook once before running any experiment notebook.**

## Imports & Configuration

In [1]:
import pathlib

import numpy as np
import pandas as pd

from data_loading_code import preprocess_pandas
from utils import stratified_split

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1
np.random.seed(SEED)

# ── Split ratios ──────────────────────────────────────────────────────────────
# Hold-out test: 10 %  →  remaining 90 % split 90/10 → train/val
# Effective split:  train ≈ 81 %  |  val ≈ 9 %  |  test = 10 %
TEST_SIZE = 0.10
VAL_SIZE  = 0.10   # fraction of the train+val pool

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = pathlib.Path('../data')
SPLITS_DIR = DATA_DIR / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RAW_1K  = DATA_DIR / 'amazon_cells_labelled.txt'
RAW_25K = DATA_DIR / 'amazon_cells_labelled_LARGE_25K.txt'

print(f'Splits will be saved to: {SPLITS_DIR.resolve()}')

Splits will be saved to: C:\Users\Oscar\Projects\D7047E\Lab1\data\splits


## Helper — Split & Save

Encapsulates the full pipeline for a single dataset: load → preprocess → split → save.

In [2]:
def load_and_split(
    raw_path: pathlib.Path,
    prefix: str,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    seed: int = SEED,
) -> dict[str, pd.DataFrame]:
    """Load, preprocess, split, and save one dataset.

    Parameters
    ----------
    raw_path : path to the raw tab-separated .txt file
    prefix   : short label used in output filenames, e.g. '1k' or '25k'
    test_size: fraction of the full dataset held out as test set
    val_size : fraction of the train+val pool used for validation
    seed     : random state for reproducible splits

    Returns
    -------
    dict with keys 'train', 'val', 'test', each a DataFrame.
    """
    # ── 1. Load ───────────────────────────────────────────────────────────────
    raw = pd.read_csv(
        raw_path,
        delimiter='\t',
        header=None,
        names=['Sentence', 'Class'],
    )
    print(f'[{prefix}] Loaded {len(raw):,} raw reviews')
    print(raw['Class'].value_counts().rename({0: 'negative', 1: 'positive'}).to_string())

    # ── 2. Preprocess ─────────────────────────────────────────────────────────
    data = preprocess_pandas(raw)
    print(f'[{prefix}] Preprocessing complete — {len(data):,} rows retained\n')

    # ── 3. Split — stratified to preserve class balance ───────────────────────
    splits = stratified_split(
        data=data,
        label_col='Class',
        test_size=test_size,
        val_size=val_size,
        seed=seed
    )

    # ── 4. Save ───────────────────────────────────────────────────────────────
    for name, df in splits.items():
        out = SPLITS_DIR / f'{prefix}_{name}.csv'
        df.to_csv(out, index=False)
        print(f'  Saved → {out}')

    return splits

## 1 K Dataset

In [3]:
splits_1k = load_and_split(RAW_1K, prefix='1k')

[1k] Loaded 1,000 raw reviews
Class
negative    500
positive    500
[1k] Preprocessing complete — 1,000 rows retained

Stratified split — total: 1,000  |  seed=1
  train:     810  (81.0 %)  [0=405  1=405]
  val  :      90  ( 9.0 %)  [0=45  1=45]
  test :     100  (10.0 %)  [0=50  1=50]
  Saved → ..\data\splits\1k_train.csv
  Saved → ..\data\splits\1k_val.csv
  Saved → ..\data\splits\1k_test.csv


In [4]:
splits_1k['train'].head()

,Sentence,Class
0,uncomfortable ear dont use lg vx env,0
1,two years left contract hate phone,0
2,im still infatuated phone,1
3,exchanged sony ericson za im pretty happy deci...,1
4,didnt think instructions provided helpful,0


## 25 K Dataset

In [5]:
splits_25k = load_and_split(RAW_25K, prefix='25k')

[25k] Loaded 25,000 raw reviews
Class
positive    15116
negative     9884
[25k] Preprocessing complete — 25,000 rows retained

Stratified split — total: 25,000  |  seed=1
  train:  20,250  (81.0 %)  [0=8,006  1=12,244]
  val  :   2,250  ( 9.0 %)  [0=890  1=1,360]
  test :   2,500  (10.0 %)  [0=988  1=1,512]
  Saved → ..\data\splits\25k_train.csv
  Saved → ..\data\splits\25k_val.csv
  Saved → ..\data\splits\25k_test.csv


In [6]:
splits_25k['train'].head()

,Sentence,Class
0,place orders involves seller respond emails re...,0
1,without blue oysters musicthe sound track lost...,0
2,set took test shot result better expected prof...,1
3,wheres sex plamer stop using term little one h...,0
4,book provides nice review office helpful learn...,1


## Sanity Checks

Verify that all six files exist and have the expected row counts.

In [7]:
print('Split file summary\n' + '-' * 48)
for f in sorted(SPLITS_DIR.glob('*.csv')):
    df = pd.read_csv(f)
    neg = (df['Class'] == 0).sum()
    pos = (df['Class'] == 1).sum()
    print(f'{f.name:<20}  rows={len(df):>6,}  neg={neg:>5,}  pos={pos:>5,}')

print('\nAll files verified.')

Split file summary
------------------------------------------------
1k_test.csv           rows=   100  neg=   50  pos=   50
1k_train.csv          rows=   810  neg=  405  pos=  405
1k_val.csv            rows=    90  neg=   45  pos=   45
25k_test.csv          rows= 2,500  neg=  988  pos=1,512
25k_train.csv         rows=20,250  neg=8,006  pos=12,244
25k_val.csv           rows= 2,250  neg=  890  pos=1,360

All files verified.


## How to Load the Splits in Other Notebooks

```python
import pandas as pd

# ── 1 K splits ────────────────────────────────────────────────────────────────
train_1k = pd.read_csv('../data/splits/1k_train.csv')
val_1k   = pd.read_csv('../data/splits/1k_val.csv')
test_1k  = pd.read_csv('../data/splits/1k_test.csv')

X_train, y_train = train_1k['Sentence'].values, train_1k['Class'].values
X_val,   y_val   = val_1k['Sentence'].values,   val_1k['Class'].values
X_test,  y_test  = test_1k['Sentence'].values,  test_1k['Class'].values

# ── 25 K splits ───────────────────────────────────────────────────────────────
train_25k = pd.read_csv('../data/splits/25k_train.csv')
val_25k   = pd.read_csv('../data/splits/25k_val.csv')
test_25k  = pd.read_csv('../data/splits/25k_test.csv')
```

> The text is already preprocessed (lowercased, stopwords removed, etc.) — do **not** call `preprocess_pandas` again in the experiment notebooks.